In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [2]:
# !uv add transformers==4.56.2
!uv pip install --no-deps trl==0.22.2

Using Python 3.12.13 environment at: /Users/vasim/Programming/ai-engineering/fine-tuning-qwen/.venv
Checked 1 package in 3ms


### Unsloth

In [3]:
from unsloth import FastLanguageModel
import torch

fourbit_models = [
    "unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit", # Qwen 14B 2x faster
    "unsloth/Qwen3-4B-Thinking-2507-unsloth-bnb-4bit",
    "unsloth/Qwen3-8B-unsloth-bnb-4bit",
    "unsloth/Qwen3-14B-unsloth-bnb-4bit",
    "unsloth/Qwen3-32B-unsloth-bnb-4bit",
    "unsloth/Qwen3-4B-Instruct-2507"

    # 4bit dynamic quants for superior accuracy and low memory use
    "unsloth/gemma-3-12b-it-unsloth-bnb-4bit",
    "unsloth/Phi-4",
    "unsloth/Llama-3.1-8B",
    "unsloth/Llama-3.2-3B",
    "unsloth/orpheus-3b-0.1-ft-unsloth-bnb-4bit" # [NEW] We support TTS models!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-unsloth-bnb-4bit",
    max_seq_length = 65536, # 64k # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

/Users/vasim/Programming/ai-engineering/fine-tuning-qwen/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Unsloth: mlx-lm cannot load bitsandbytes 4-bit weights; loading base 'unsloth/Qwen2.5-1.5B' and applying MLX 4-bit quantization instead of 'unsloth/Qwen2.5-1.5B-unsloth-bnb-4bit'.


Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 281706.99it/s]


Unsloth: Loading unsloth/Qwen2.5-1.5B via mlx-lm (runtime 4-bit affine quantization)...


Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 288158.29it/s]


[INFO] Quantized model with 6.240 bits per weight.
Unsloth: Quantized text model to 4-bit affine.


We now add LoRA adapters so we only need to update a small amount of parameters!

In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth: LoRA applied — 36,929,536 trainable params (7.77% of 475,166,208 total)


<a name="Data"></a>
### Data Prep
We now use the `Qwen-3` format for conversation style finetunes. We use [Maxime Labonne's FineTome-100k](https://huggingface.co/datasets/mlabonne/FineTome-100k) dataset in ShareGPT style. Qwen-3 renders multi turn conversations like below:

```
<|im_start|>user
Hello!<|im_end|>
<|im_start|>assistant
Hey there!<|im_end|>

```
We use our `get_chat_template` function to get the correct chat template. We support `zephyr, chatml, mistral, llama, alpaca, vicuna, vicuna_old, phi3, llama3, phi4, qwen2.5, gemma3` and more.

In [5]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen3-instruct",
)

In [6]:
# tokenizer

In [7]:
from datasets import load_dataset

# dataset = load_dataset("mlabonne/FineTome-100k", split = "train")
dataset = load_dataset("hiyouga/glaive-function-calling-v2-sharegpt", split = "train", cache_dir="../data/")

In [8]:
dataset[0:10]

{'tools': ['[{"name": "get_exchange_rate", "description": "Get the exchange rate between two currencies", "parameters": {"type": "object", "properties": {"base_currency": {"type": "string", "description": "The currency to convert from"}, "target_currency": {"type": "string", "description": "The currency to convert to"}}, "required": ["base_currency", "target_currency"]}}]',
  '[{"name": "get_news_headlines", "description": "Get the latest news headlines", "parameters": {"type": "object", "properties": {"country": {"type": "string", "description": "The country for which to fetch news"}}, "required": ["country"]}}]',
  '[{"name": "generate_password", "description": "Generate a random password", "parameters": {"type": "object", "properties": {"length": {"type": "integer", "description": "The length of the password"}, "include_symbols": {"type": "boolean", "description": "Whether to include symbols in the password"}}, "required": ["length"]}}, {"name": "create_task", "description": "Create

**References used for the data-processing pipeline**

- [Qwen 2.5 official tool-calling docs](https://qwen.readthedocs.io/en/latest/framework/Function_call.html) — canonical schema for `tool_calls` and tool responses
- [Qwen 2.5 Instruct `tokenizer_config.json`](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct/raw/main/tokenizer_config.json) — the exact `chat_template` Jinja string the model was trained with (we matched this verbatim)
- [Qwen 2.5-1.5B-Instruct model card](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct) — base model reference
- [Glaive function calling v2 (ShareGPT)](https://huggingface.co/datasets/hiyouga/glaive-function-calling-v2-sharegpt) — source dataset
- [Unsloth `standardize_data_formats` source](https://github.com/unslothai/unsloth/blob/main/unsloth_zoo/dataset_utils.py) — the function that rejects `function_call`/`observation` roles
- [Hugging Face `apply_chat_template` docs](https://huggingface.co/docs/transformers/main/en/chat_templating#advanced-tool-use) — `tools=` kwarg usage


We now use `standardize_data_formats` to try converting datasets to the correct format for finetuning purposes!

**Custom preprocessing for Glaive function-calling dataset**

The `hiyouga/glaive-function-calling-v2-sharegpt` dataset uses roles `human`, `gpt`, `function_call`, and `observation` — but `standardize_data_formats` only knows about `system`/`user`/`assistant` aliases. We first convert `function_call` -> assistant message with `tool_calls` and `observation` -> `tool` message with `tool_call_id` (Qwen 2.5 tool-calling format), so the chat template renders them correctly.


In [9]:
import json, uuid

def to_qwen_tool_format(example):
    """Convert glaive-function-calling ShareGPT format to Qwen 2.5 tool-calling format.

    Source order:  human -> function_call -> observation -> gpt(answer)
    Target order:  user -> assistant(tool_calls) -> tool -> assistant(answer)

    We SPLIT the post-observation gpt message so the tool_calls come BEFORE
    the tool response, not after. The SFT model needs to learn to emit
    `<tool_call>...</tool_call>` first, then receive `<tool_response>`, then give
    the final answer — that's the format Qwen 2.5 was trained on.

    The 'tools' field is left as a raw JSON string (we don't pre-parse it here)
    because the `datasets` library cannot infer a single PyArrow schema for the
    heterogeneous tool-list column (each row has different parameter shapes).
    The downstream `formatting_prompts_func` parses it before passing to
    `tokenizer.apply_chat_template(tools=...)`.
    """
    convos = example["conversations"]
    new_convo = []
    pending_tool_call_msgs = []

    for msg in convos:
        role = msg["from"]
        text = msg["value"]
        if role == "human":
            new_convo.append({"role": "user", "content": text})
        elif role == "gpt":
            if pending_tool_call_msgs:
                tool_calls = [m["args"] for m in pending_tool_call_msgs if m["role"] == "tool_call"]
                tool_responses = [
                    (m["call_id"], m["content"])
                    for m in pending_tool_call_msgs
                    if m["role"] == "tool_response"
                ]
                if tool_calls:
                    new_convo.append({
                        "role": "assistant",
                        "content": "",
                        "tool_calls": tool_calls,
                    })
                for call_id, content in tool_responses:
                    new_convo.append({
                        "role": "tool",
                        "tool_call_id": call_id,
                        "content": content,
                    })
                pending_tool_call_msgs = []
            new_convo.append({"role": "assistant", "content": text})
        elif role == "function_call":
            try:
                fc = json.loads(text)
                call_id = f"call_{uuid.uuid4().hex[:24]}"
                arg_str = json.dumps(fc.get("arguments", {}), ensure_ascii=False)
                pending_tool_call_msgs.append({
                    "role": "tool_call",
                    "call_id": call_id,
                    "args": {
                        "id": call_id,
                        "type": "function",
                        "function": {
                            "name": fc.get("name", ""),
                            "arguments": arg_str,
                        },
                    },
                })
            except Exception:
                pass
        elif role == "observation":
            if pending_tool_call_msgs:
                latest = None
                for item in reversed(pending_tool_call_msgs):
                    if item["role"] == "tool_call":
                        latest = item
                        break
                if latest:
                    pending_tool_call_msgs.append({
                        "role": "tool_response",
                        "call_id": latest["call_id"],
                        "content": text,
                    })

    if pending_tool_call_msgs:
        tool_calls = [m["args"] for m in pending_tool_call_msgs if m["role"] == "tool_call"]
        tool_responses = [
            (m["call_id"], m["content"])
            for m in pending_tool_call_msgs
            if m["role"] == "tool_response"
        ]
        if tool_calls:
            new_convo.append({"role": "assistant", "content": "", "tool_calls": tool_calls})
        for call_id, content in tool_responses:
            new_convo.append({"role": "tool", "tool_call_id": call_id, "content": content})

    # Keep 'tools' as a raw JSON string to avoid PyArrow schema-inference failures
    # caused by heterogeneous tool definitions across rows.
    raw_tools_str = example.get("tools", "") or ""
    return {"conversations": new_convo, "tools": raw_tools_str}

# Remove only the original 'conversations' field (tools must be preserved as a string).
dataset = dataset.map(
    to_qwen_tool_format,
    remove_columns=[c for c in dataset.column_names if c != "tools"],
)
print("Sample after conversion:")
print(json.dumps(dataset[0], indent=2, default=str)[:2000])


Sample after conversion:
{
  "tools": "[{\"name\": \"get_exchange_rate\", \"description\": \"Get the exchange rate between two currencies\", \"parameters\": {\"type\": \"object\", \"properties\": {\"base_currency\": {\"type\": \"string\", \"description\": \"The currency to convert from\"}, \"target_currency\": {\"type\": \"string\", \"description\": \"The currency to convert to\"}}, \"required\": [\"base_currency\", \"target_currency\"]}}]",
  "conversations": [
    {
      "role": "user",
      "content": "Can you book a flight for me from New York to London?",
      "tool_calls": null,
      "tool_call_id": null
    },
    {
      "role": "assistant",
      "content": "I'm sorry, but I don't have the capability to book flights. My current function allows me to get the exchange rate between two currencies. If you need help with that, feel free to ask!",
      "tool_calls": null,
      "tool_call_id": null
    }
  ]
}


Let's see how row 100 looks like!

In [10]:
dataset[100]

{'tools': '[{"name": "generate_password", "description": "Generate a random password", "parameters": {"type": "object", "properties": {"length": {"type": "integer", "description": "The length of the password"}}, "required": ["length"]}}, {"name": "check_email", "description": "Check if an email address is valid", "parameters": {"type": "object", "properties": {"email": {"type": "string", "description": "The email address to check"}}, "required": ["email"]}}]',
 'conversations': [{'role': 'user',
   'content': 'Hi, I need a new password. Can you generate one for me?',
   'tool_calls': None,
   'tool_call_id': None},
  {'role': 'assistant',
   'content': 'Of course, I can help with that. How long would you like your password to be?',
   'tool_calls': None,
   'tool_call_id': None},
  {'role': 'user',
   'content': 'I would like it to be 12 characters long.',
   'tool_calls': None,
   'tool_call_id': None},
  {'role': 'assistant',
   'content': '',
   'tool_calls': [{'id': 'call_62503e4c5

In [11]:
dataset[202]

{'tools': '[{"name": "calculate_tip", "description": "Calculate the tip amount based on the bill amount and tip percentage", "parameters": {"type": "object", "properties": {"bill_amount": {"type": "number", "description": "The total bill amount"}, "tip_percentage": {"type": "number", "description": "The tip percentage"}}, "required": ["bill_amount", "tip_percentage"]}}, {"name": "calculate_age", "description": "Calculate the age based on birthdate", "parameters": {"type": "object", "properties": {"birthdate": {"type": "string", "description": "The birthdate of the person"}}, "required": ["birthdate"]}}]',
 'conversations': [{'role': 'user',
   'content': 'Hi, I need help calculating the tip for my bill. The total bill amount is $100 and I want to leave a 15% tip.',
   'tool_calls': None,
   'tool_call_id': None},
  {'role': 'assistant',
   'content': '',
   'tool_calls': [{'id': 'call_2cd7a8808ff94ff8b03896ce',
     'type': 'function',
     'function': {'name': 'calculate_tip',
      '

We now have to apply the chat template for `Qwen-3` onto the conversations, and save it to `text`.

In [12]:
def _parse_tools(tools_str):
    """Parse the JSON-string tools column into a list[dict] of OpenAI-envelope tools.
    Empty list if '[]' or unparseable."""
    if not tools_str or not isinstance(tools_str, str):
        return []
    s = tools_str.strip()
    if not s or s == "[]":
        return []
    try:
        parsed = json.loads(s)
    except Exception:
        return []
    if not isinstance(parsed, list):
        return []
    out = []
    for t in parsed:
        if not isinstance(t, dict):
            continue
        if "function" in t and isinstance(t["function"], dict):
            if "type" not in t:
                t["type"] = "function"
            out.append(t)
        elif "name" in t:
            out.append({
                "type": "function",
                "function": {
                    "name": t.get("name", ""),
                    "description": t.get("description", ""),
                    "parameters": t.get("parameters", {"type": "object", "properties": {}}),
                },
            })
    return out

def formatting_prompts_func(examples):
   convos = examples["conversations"]
   tools_batch = [_parse_tools(t) for t in examples.get("tools", [""] * len(convos))]
   texts = [
       tokenizer.apply_chat_template(
           convo,
           tools=(tools if tools else None),
           tokenize=False,
           add_generation_prompt=False,
       )
       for convo, tools in zip(convos, tools_batch)
   ]
   return { "text": texts }

dataset = dataset.map(formatting_prompts_func, batched=True)


Let's see how the chat template did!

In [13]:
dataset[102]['text']

'<|im_start|>system\n# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>\n{"type": "function", "function": {"name": "calculate_area", "description": "Calculate the area of a shape", "parameters": {"type": "object", "properties": {"shape": {"type": "string", "description": "The shape to calculate the area for"}, "measurements": {"type": "object", "properties": {"length": {"type": "number", "description": "The length of the shape"}, "width": {"type": "number", "description": "The width of the shape"}}, "required": ["length", "width"]}}, "required": ["shape", "measurements"]}}}\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{"name": <function-name>, "arguments": <args-json-object>}\n</tool_call><|im_end|>\n<|im_start|>user\nI need to calculate the area of a rectangle. The length is

**Debug: verify `<tool_call>` is in the rendered training text**

SFT teaches the model by imitation. For the model to learn when to emit a `<tool_call>` block, that block must appear in the rendered training text inside an assistant turn. This cell prints the parsed conversation (with `tool_calls` attached) and the rendered text side by side so we can verify both the preprocessing and the chat template are working correctly. See the [Qwen 2.5 tool-calling docs](https://qwen.readthedocs.io/en/latest/framework/Function_call.html) for the expected format.


In [14]:
# Debug: show the parsed conversation AND the rendered text for row 1
# to verify the <tool_call> block is being attached and rendered correctly.
import json
print("=== Parsed conversations (row 1) ===")
print(json.dumps(dataset[1]["conversations"], indent=2)[:2000])
print()
print("=== Tools (row 1) ===")
print(json.dumps(dataset[1]["tools"], indent=2)[:1500])
print()
print("=== Rendered text (row 1) ===")
print(dataset[1]["text"])


=== Parsed conversations (row 1) ===
[
  {
    "role": "user",
    "content": "Can you tell me the latest news headlines for the United States?",
    "tool_calls": null,
    "tool_call_id": null
  },
  {
    "role": "assistant",
    "content": "",
    "tool_calls": [
      {
        "id": "call_975e10f8490245b8922317cf",
        "type": "function",
        "function": {
          "name": "get_news_headlines",
          "arguments": "{\"country\": \"United States\"}"
        }
      }
    ],
    "tool_call_id": null
  },
  {
    "role": "tool",
    "content": "{\"headlines\": [\"Biden announces new vaccine mandates\", \"Hurricane Ida devastates Louisiana\", \"Apple unveils new iPhone\", \"NASA's Perseverance rover collects first Mars rock sample\"]}",
    "tool_calls": null,
    "tool_call_id": "call_975e10f8490245b8922317cf"
  },
  {
    "role": "assistant",
    "content": "Here are the latest news headlines for the United States:\n1. Biden announces new vaccine mandates\n2. Hurricane 

In [15]:
print(dataset[1]['text'])

<|im_start|>system
# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"type": "function", "function": {"name": "get_news_headlines", "description": "Get the latest news headlines", "parameters": {"type": "object", "properties": {"country": {"type": "string", "description": "The country for which to fetch news"}}, "required": ["country"]}}}
</tools>

For each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:
<tool_call>
{"name": <function-name>, "arguments": <args-json-object>}
</tool_call><|im_end|>
<|im_start|>user
Can you tell me the latest news headlines for the United States?<|im_end|>
<|im_start|>assistant
<tool_call>
{"name": "get_news_headlines", "arguments": {"country": "United States"}}
</tool_call><|im_end|>
<|im_start|>user
<tool_response>
{"headlines": ["Biden announces new vaccine mandates", "Hurricane Ida 

In [16]:
print(dataset[22]['text'])

<|im_start|>system
# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"type": "function", "function": {"name": "search_books", "description": "Search for books based on title, author, or genre", "parameters": {"type": "object", "properties": {"title": {"type": "string", "description": "The title of the book"}, "author": {"type": "string", "description": "The author of the book"}, "genre": {"type": "string", "description": "The genre of the book"}}, "required": []}}}
</tools>

For each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:
<tool_call>
{"name": <function-name>, "arguments": <args-json-object>}
</tool_call><|im_end|>
<|im_start|>user
I am looking for a book but I can't remember the title. The author's name is George Orwell.<|im_end|>
<|im_start|>assistant
<tool_call>
{"name": "search_books", "arguments": {"auth